In [1]:
import os
import pandas as pd  # to load data into DataFrame
import numpy as np   # to deal with arrays and numerical values
import datetime #from datetime import date
import traceback
from ctREFPROP.ctREFPROP import REFPROPFunctionLibrary

import sqlite3

from datetime import datetime, timedelta # to get date and time # to represent the difference between two dates or times

In [2]:
scenario_ambient_temp = [-0.3, -0.1, 0.1] #[-0.3, -0.17, -0.03, 0.1] 230912 isko offset을 넣기 위해 -0.17과 -0.03을 빼고 -0.1을 넣음
scenario_delta_temp = [-0.3, -0.15, -0.0] # [-0.3, -0.15, 0.0] [-0.3, -0.2, -0.1, 0.0]
scenario_offset = [0, 0.2] # 230912 isko 

In [3]:
df_policy = pd.read_csv('C:/KHU_DRL/1st_policy.csv', header = None)
# 이 csv를 입력하는 파일을 만들어야 한다
coef_amb = df_policy.iloc[0,0] #scenario_ambient_temp[df_policy.iloc[0,0]]
code_delta = df_policy.iloc[0,1] #scenario_delta_temp[df_policy.iloc[0,1]]
offset = df_policy.iloc[0,2] #isko 230912

In [4]:
def target_press_by_scenario(coef_amb, code_delta, offset, ambient_temp, set_temp, inner_temp, op_status): #isko 230912 offset 추가
    target = coef_amb*(ambient_temp-30) + code_delta*(inner_temp-set_temp) + offset + 8 #35 -> 30, isko 230912 offset 추가
    target = max(min(target, 9.5), 6.5)
    if op_status==0:
        target = 0
    else:
        pass
    
    return round(target,1)

In [5]:
#====================================================================================
# Transform the time of day and year into sine and cosine signals
day = 60 * 60 * 24
def get_sine_time(time):
    seconds = int(round(time.timestamp()))
    time_sine = np.sin(seconds * (2 * np.pi / day))

    return time_sine


def get_cosine_time(time):
    seconds = int(round(time.timestamp()))
    time_cosine = np.cos(seconds * (2 * np.pi / day))

    return time_cosine


year = (365.2425)*day
def get_sin_year(Timestamp):
    seconds_y = int(round(Timestamp.timestamp()))
    year_sine = np.sin(seconds_y * (2 * np.pi / year))

    return year_sine

def get_cos_year(Timestamp):
    seconds_y = int(round(Timestamp.timestamp()))
    year_cosine = np.cos(seconds_y * (2 * np.pi / year))

    return year_cosine

In [6]:
# Connect to the Db
connection = sqlite3.connect('C:/KHU_DRL/NED_SQLite.db')

In [7]:
sql = "SELECT * FROM odu1_data ORDER BY timestamp DESC LIMIT 10 " #가장 최근 데이타 10개 사용(10분)

odu1_data = pd.read_sql_query(sql, connection)

odu1_data = odu1_data.sort_values(by='timestamp')
odu1_data['timestamp'] = pd.to_datetime(odu1_data['timestamp'])

# operation mode encoding
odu1_data['IDU0_onoff'].replace({"운전 On": 1, "운전 Off": 0}, inplace=True)
odu1_data['IDU1_onoff'].replace({"운전 On": 1, "운전 Off": 0}, inplace=True)
odu1_data['IDU2_onoff'].replace({"운전 On": 1, "운전 Off": 0}, inplace=True)
odu1_data['IDU3_onoff'].replace({"운전 On": 1, "운전 Off": 0}, inplace=True)

odu1_data['ODU_onoff'].replace({"일반 운전": 1, "정지": 0}, inplace=True)

# Capacity weighted calculation

# odu1_data.loc[(odu1_data["IDU0_onoff"] == 0) & (odu1_data["IDU1_onoff"] == 0) &
#         (odu1_data["IDU2_onoff"] == 0) & (odu1_data["IDU3_onoff"] == 0), ["IDU0_onoff",
#         "IDU1_onoff", "IDU2_onoff", "IDU3_onoff"]] = 1

i_case = len(odu1_data)


In [8]:
odu1_data.columns

Index(['timestamp', 'IDU0_onoff', 'IDU1_onoff', 'IDU2_onoff', 'IDU3_onoff',
       'IDU0_settemp', 'IDU1_settemp', 'IDU2_settemp', 'IDU3_settemp',
       'IDU0_intemp', 'IDU1_intemp', 'IDU2_intemp', 'IDU3_intemp', 'IDU0_capa',
       'IDU1_capa', 'IDU2_capa', 'IDU3_capa', 'ODU_outtemp', 'ODU_targetpress',
       'ODU_highpress', 'ODU_power', 'ODU_onoff'],
      dtype='object')

In [9]:


for i in range(0, i_case):
    running_capa = ((odu1_data.loc[i, 'IDU0_onoff'] * odu1_data.loc[i, 'IDU0_capa']) +
            (odu1_data.loc[i, 'IDU1_onoff'] * odu1_data.loc[i, 'IDU1_capa']) +
            (odu1_data.loc[i, 'IDU2_onoff'] * odu1_data.loc[i, 'IDU2_capa']) +
            (odu1_data.loc[i, 'IDU3_onoff'] * odu1_data.loc[i, 'IDU3_capa']))

    if running_capa == 0:
        odu1_data.loc[i, "IAT"] = (( odu1_data.loc[i, 'IDU0_intemp'] * odu1_data.loc[i, 'IDU0_capa'])+
                    ( odu1_data.loc[i, 'IDU1_intemp'] * odu1_data.loc[i, 'IDU1_capa']) +
                    ( odu1_data.loc[i, 'IDU2_intemp'] * odu1_data.loc[i, 'IDU2_capa']) +
                    ( odu1_data.loc[i, 'IDU3_intemp'] * odu1_data.loc[i, 'IDU3_capa'])) / \
                    ( odu1_data.loc[i, 'IDU0_capa'] + odu1_data.loc[i, 'IDU1_capa'] + odu1_data.loc[i, 'IDU2_capa'] + odu1_data.loc[i, 'IDU3_capa'] )           
        odu1_data.loc[i, 'ST'] = odu1_data.loc[i, "IAT"]

    else:           
        odu1_data.loc[i, 'ST'] = ((odu1_data.loc[i, 'IDU0_onoff'] * odu1_data.loc[i, 'IDU0_settemp'] * odu1_data.loc[i, 'IDU0_capa'])+
                (odu1_data.loc[i, 'IDU1_onoff'] * odu1_data.loc[i, 'IDU1_settemp'] * odu1_data.loc[i, 'IDU1_capa']) +
                (odu1_data.loc[i, 'IDU2_onoff'] * odu1_data.loc[i, 'IDU2_settemp'] * odu1_data.loc[i, 'IDU2_capa']) +
                (odu1_data.loc[i, 'IDU3_onoff'] * odu1_data.loc[i, 'IDU3_settemp'] * odu1_data.loc[i, 'IDU3_capa'])) / running_capa
        odu1_data.loc[i, "IAT"] = ((odu1_data.loc[i, 'IDU0_onoff'] * odu1_data.loc[i, 'IDU0_intemp'] * odu1_data.loc[i, 'IDU0_capa'])+
                    (odu1_data.loc[i, 'IDU1_onoff'] * odu1_data.loc[i, 'IDU1_intemp'] * odu1_data.loc[i, 'IDU1_capa']) +
                    (odu1_data.loc[i, 'IDU2_onoff'] * odu1_data.loc[i, 'IDU2_intemp'] * odu1_data.loc[i, 'IDU2_capa']) +
                    (odu1_data.loc[i, 'IDU3_onoff'] * odu1_data.loc[i, 'IDU3_intemp'] * odu1_data.loc[i, 'IDU3_capa'])) / running_capa

odu1_data_r = odu1_data[['timestamp', 'ODU_outtemp', 'ODU_targetpress','ODU_highpress', 'ODU_power', 'ODU_onoff', 'ST', 'IAT']]

# Instantanious values for DRL training
odu1_data_r = odu1_data_r.set_index('timestamp')
odu1_data_avg = odu1_data_r.mean()

target_press = target_press_by_scenario(coef_amb, code_delta, offset, odu1_data_avg['ODU_outtemp'], odu1_data_avg['ST'], odu1_data_avg['IAT'], 1) #230912 isko

In [10]:
i_cali = 0
i_operation = 0
for i in range(0,4):
    for i_line in range(0,10):
        #print(i, i_line)
        if odu1_data.loc[i_line, 'IDU' + str(i) + '_onoff'] == 1:
            i_operation = i_operation + 1
            #print(odu1_data.loc[i_line, 'IDU' + str(i) + '_settemp'])
            if odu1_data.loc[i_line, 'IDU' + str(i) + '_intemp'] - odu1_data.loc[i_line, 'IDU' + str(i) + '_settemp'] < 1:
                i_cali = i_cali + 1
                #break
                
      

In [11]:
if i_operation != 0:
    target_press = target_press + round(i_cali / i_operation * 0.5, 1)
else:
    pass

In [12]:
cursor = connection.cursor()
cursor.execute( 'insert into odu1_drl_policy(timestamp, ODU_targetpress, ODU_onoff) values (?, ?, ?)', 
               (datetime.now().strftime('%Y-%m-%d %H:%M:%S') ,target_press, '정상'))
connection.commit()
connection.close()

In [13]:
target_press

8.7